# Notebook 3: Embedding Improvements

**Goal:** Systematically ablate text representation components and measure their impact on retrieval quality.

**Key principle:** `src/text_builders.py` is NOT modified. All variants are realized by passing different flag arguments to `dish_to_rich_text()`. The reviewer sees baseline and improvement in the same notebook.

**Experiments:**
1. Remove recipe text (`include_recipe=False`)
2. Remove discrete macro tokens (`include_macro_tokens=False`)
3. Add continuous macro ratios (`include_ratios=True`)
4. Add extracted ingredients (`include_ingredients=True`)
5. Behavioral embedding from dish vectors (vs. text encoding)
6. Multi-vector retrieval
7. Cross-encoder reranking

In [ ]:
# SETUP
import os, sys
REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"
if not os.path.exists(REPO): !git clone https://github.com/{GITHUB_USER}/{REPO}.git
else: !cd {REPO} && git pull -q
!pip install -q sentence-transformers faiss-cpu pandas pyarrow matplotlib scikit-learn
sys.path.insert(0, f'/content/{REPO}/src')

from text_builders import dish_to_rich_text
from embedding_model import EmbeddingModel
from behavioral_embedding import build_behavioral_embedding_from_dishes
from multi_vector_retrieval import multi_vector_search, build_retrieval_queries
from cross_encoder_reranker import CrossEncoderReranker
from utils import evaluate_all, plot_metric_comparison, plot_cumulative_improvements

import pandas as pd, numpy as np, faiss, json
np.random.seed(42)
print("✓ Setup complete")

In [ ]:
# ============================================================
# GOOGLE DRIVE - read processed data saved by Notebook 01
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

PROCESSED_DIR = '/content/drive/MyDrive/food-recsys/processed'
print('Reading data from: ' + PROCESSED_DIR)

In [ ]:
recipes = pd.read_parquet(f'{PROCESSED_DIR}/recipes.parquet')
train = pd.read_parquet(f'{PROCESSED_DIR}/interactions_train.parquet')
test = pd.read_parquet(f'{PROCESSED_DIR}/interactions_test.parquet')

recipe_id_to_idx = {rid: i for i, rid in enumerate(recipes['id'])}
idx_to_recipe_id = {i: rid for i, rid in enumerate(recipes['id'])}

with open(f'{PROCESSED_DIR}/results.json') as f:
    all_results = json.load(f)

model = EmbeddingModel()
print(f"Model loaded: {model.model_name}")

In [ ]:
# Reuse evaluation function from NB2
def evaluate_retrieval(index, embeddings, ks=None):
    if ks is None: ks = [5, 10, 20]
    user_positives = (
        test[test['rating'] >= 4].groupby('user_id')['recipe_id'].apply(set).to_dict()
    )
    results = []
    for user_id, pos_recipes in user_positives.items():
        pos_recipes = {r for r in pos_recipes if r in recipe_id_to_idx}
        if len(pos_recipes) < 5: continue
        query_recipe = list(pos_recipes)[0]
        relevant = pos_recipes - {query_recipe}
        query_idx = recipe_id_to_idx[query_recipe]
        scores, indices = index.search(embeddings[query_idx:query_idx+1], max(ks) + 1)
        recommended = [idx_to_recipe_id[i] for i in indices[0] if i >= 0 and idx_to_recipe_id.get(i) != query_recipe]
        results.append(evaluate_all(recommended, relevant, ks=ks))
    return pd.DataFrame(results).mean().to_dict() if results else {}


def run_experiment(config_name, **flag_kwargs):
    """Build texts → encode → build index → evaluate."""
    print(f"Running: {config_name}  flags={flag_kwargs}")
    texts = [dish_to_rich_text(row.to_dict(), tags=row.get('tag_list', []), **flag_kwargs)
             for _, row in recipes.iterrows()]
    embs = model.encode(texts)
    idx = faiss.IndexFlatIP(model.dim)
    idx.add(embs)
    metrics = evaluate_retrieval(idx, embs)
    all_results[config_name] = metrics
    print(f"  P@10={metrics.get('P@10', 0):.4f}  NDCG@10={metrics.get('NDCG@10', 0):.4f}")
    return metrics, embs

print("Helper functions ready")

## Experiment 1: Remove Recipe Text

**Hypothesis:** Recipe instructions describe cooking procedures, not dish identity. Sentences like "add salt, stir for 5 minutes" contribute noise to the embedding. Removing them should improve retrieval of similar dishes.

In [ ]:
metrics_no_recipe, _ = run_experiment('no_recipe', include_recipe=False)

delta = metrics_no_recipe.get('P@10', 0) - all_results['baseline_embedding'].get('P@10', 0)
print(f"\nΔ P@10 vs baseline: {delta:+.4f}")

## Experiment 2: Remove Discrete Macro Tokens

**Hypothesis:** Discrete thresholds (HIGH_PROTEIN if protein ≥ 25g) create hard cluster boundaries in embedding space. A dish with 24g protein and one with 26g protein get different tokens but should be semantically similar. Removing these tokens should smooth out artificial boundaries.

In [ ]:
metrics_no_recipe_no_macro, _ = run_experiment(
    'no_recipe_no_macro',
    include_recipe=False,
    include_macro_tokens=False,
)

## Experiment 3: Add Continuous Macro Ratios

**Hypothesis:** Continuous ratios (protein_ratio=0.51) represent nutritional composition smoothly. This preserves semantic proximity between dishes with similar macro profiles.

In [ ]:
metrics_with_ratios, _ = run_experiment(
    'no_recipe_no_macro_with_ratios',
    include_recipe=False,
    include_macro_tokens=False,
    include_ratios=True,
)

## Experiment 4: Add Extracted Ingredients

**Hypothesis:** Ingredient lists signal dish identity without procedural noise. "chicken, lemon, garlic" is more informative than the full recipe instruction.

In [ ]:
metrics_full_improved, embs_full_improved = run_experiment(
    'full_improved',
    include_recipe=False,
    include_macro_tokens=False,
    include_ratios=True,
    include_ingredients=True,
)

## Experiment 5: Behavioral Embedding from Dish Vectors

**Hypothesis:** Encoding user behavior as text ("STRONGLY PREFERS: Grilled Chicken") introduces a semantic gap — this text lives in a different region of embedding space than the dish embedding itself. Aggregating actual dish vectors eliminates this gap.

In [ ]:
# Build dish embedding map for behavioral aggregation
dish_emb_map = {idx_to_recipe_id[i]: embs_full_improved[i] for i in range(len(embs_full_improved))}

# Build behavioral embeddings for test users (from their TRAIN positives)
user_train_positives = (
    train[train['rating'] >= 4].groupby('user_id')['recipe_id'].apply(list).to_dict()
)

# Evaluate: for each test user, use their behavioral embedding as query
user_test_positives = (
    test[test['rating'] >= 4].groupby('user_id')['recipe_id'].apply(set).to_dict()
)

idx_full = faiss.IndexFlatIP(model.dim)
idx_full.add(embs_full_improved)

from datetime import datetime, timezone
results_behavioral = []
for user_id, pos_train in user_train_positives.items():
    pos_test = user_test_positives.get(user_id, set())
    pos_test = {r for r in pos_test if r in recipe_id_to_idx}
    if len(pos_test) < 3: continue

    # Build behavioral embedding from train orders
    orders = [{'dish_id': rid, 'ordered_at': datetime.now(timezone.utc)} for rid in pos_train if rid in dish_emb_map]
    beh_emb = build_behavioral_embedding_from_dishes(dish_emb_map, recent_orders=orders)
    if beh_emb is None: continue

    q = beh_emb.reshape(1, -1).astype(np.float32)
    scores, indices = idx_full.search(q, 25)
    recommended = [idx_to_recipe_id[i] for i in indices[0] if i >= 0]
    results_behavioral.append(evaluate_all(recommended, pos_test, ks=[5, 10, 20]))

metrics_behavioral = pd.DataFrame(results_behavioral).mean().to_dict() if results_behavioral else {}
all_results['behavioral_dish_vectors'] = metrics_behavioral
print(f"Behavioral (dish vectors) P@10: {metrics_behavioral.get('P@10', 0):.4f}")

## Experiment 6: Multi-Vector Retrieval

In [ ]:
from multi_vector_retrieval import multi_vector_search

# Evaluate multi-vector: query with both the item embedding AND a nutritional query
results_multi = []
for user_id, pos_recipes in user_test_positives.items():
    pos_recipes = {r for r in pos_recipes if r in recipe_id_to_idx}
    if len(pos_recipes) < 5: continue
    query_recipe = list(pos_recipes)[0]
    relevant = pos_recipes - {query_recipe}
    query_idx = recipe_id_to_idx[query_recipe]

    # Multi-vector: item embedding + high-protein query variant
    hp_text = "high protein low calorie healthy dish"
    hp_emb = model.encode([hp_text])[0]
    queries = [embs_full_improved[query_idx], hp_emb]

    ranked = multi_vector_search(queries, idx_full, top_k=20, strategy='multi', merge='rrf')
    recommended = [item_id for item_id, _ in ranked if item_id != query_recipe]
    results_multi.append(evaluate_all(recommended, relevant, ks=[5, 10, 20]))

metrics_multi = pd.DataFrame(results_multi).mean().to_dict() if results_multi else {}
all_results['multi_vector_rrf'] = metrics_multi
print(f"Multi-vector (RRF) P@10: {metrics_multi.get('P@10', 0):.4f}")

## Experiment 7: Cross-Encoder Reranking

In [ ]:
reranker = CrossEncoderReranker()
id_to_text = {row['id']: dish_to_rich_text(row.to_dict(), tags=row.get('tag_list', []),
              include_recipe=False, include_macro_tokens=False, include_ratios=True, include_ingredients=True)
              for _, row in recipes.iterrows()}

results_ce = []
for user_id, pos_recipes in list(user_test_positives.items())[:200]:  # sample for speed
    pos_recipes = {r for r in pos_recipes if r in recipe_id_to_idx}
    if len(pos_recipes) < 5: continue
    query_recipe = list(pos_recipes)[0]
    relevant = pos_recipes - {query_recipe}
    query_idx = recipe_id_to_idx[query_recipe]

    # Retrieve top-50 with bi-encoder
    scores, indices = idx_full.search(embs_full_improved[query_idx:query_idx+1], 51)
    candidates_50 = [(idx_to_recipe_id[i], id_to_text.get(idx_to_recipe_id[i], ''))
                     for i in indices[0] if i >= 0 and idx_to_recipe_id.get(i) != query_recipe][:50]

    # Rerank with cross-encoder → top-10
    query_text = id_to_text.get(query_recipe, '')
    reranked = reranker.rerank(query_text, candidates_50, top_k=10,
                               fallback_scores=scores[0][1:51].tolist())
    recommended = [item_id for item_id, _ in reranked]
    results_ce.append(evaluate_all(recommended, relevant, ks=[5, 10]))

metrics_ce = pd.DataFrame(results_ce).mean().to_dict() if results_ce else {}
all_results['cross_encoder_rerank'] = metrics_ce
print(f"Cross-encoder rerank P@10: {metrics_ce.get('P@10', 0):.4f}")

## Cumulative Results

In [ ]:
ordered_configs = [
    'baseline_embedding',
    'no_recipe',
    'no_recipe_no_macro',
    'no_recipe_no_macro_with_ratios',
    'full_improved',
    'behavioral_dish_vectors',
    'multi_vector_rrf',
    'cross_encoder_rerank',
]
labels = [
    'Baseline', '- recipe', '- macro_tokens',
    '+ ratios', '+ ingredients', '+ behavioral_vecs',
    '+ multi_vector', '+ cross_encoder',
]

results_ordered = [all_results.get(c, {}) for c in ordered_configs]
plot_cumulative_improvements(results_ordered, 'P@10', labels, title='Cumulative P@10 Improvement')

# Summary table
summary = pd.DataFrame({lab: all_results.get(cfg, {}) for lab, cfg in zip(labels, ordered_configs)}).T
print("\n=== Summary ===")
print(summary[['P@5', 'P@10', 'NDCG@10', 'MRR']].round(4).to_string())

# Save all results
with open(f'{PROCESSED_DIR}/results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print("\n✓ Results saved")